# Question 4: ETL with R and PostgreSQL
Use case: compare the Eduvos programmes (faculty, type, learning modes and start date) from the qualifications page.

In [5]:
library(rvest)
library(httr)
library(stringr)

## Q4.1 Extract: scrape the programme cards and use regex

In [6]:
ua <- user_agent("EduvosStudentProject/1.0 (non-commercial, educational)")
Sys.sleep(2)
page <- read_html(GET("https://www.eduvos.com/qualifications/", ua))
cards <- html_elements(page, "article.qf-card")            # one card per programme

raw <- data.frame(
  title          = html_text2(html_element(cards, ".qf-card-title")),
  faculty        = html_attr(cards, "data-faculty"),
  programme_type = html_attr(cards, "data-programme-type"),
  learning_modes = html_attr(cards, "data-learning-mode"),
  description    = html_text2(html_element(cards, ".qf-card-description")),
  start_text     = html_text2(html_element(cards, ".qf-start-date"))
)

In [7]:
# regex: pull the date out of "Start date: 15 February 2027"
raw$start_date <- str_extract(raw$start_text, "\\d{1,2} [A-Za-z]+ \\d{4}")
# regex: count the learning modes in the text (numerical attribute, inferred)
raw$n_modes <- str_count(raw$learning_modes, "(?i)full[- ]time|part[- ]time|remote learning")
# regex: count the words of the description (numerical attribute, inferred)
raw$desc_words <- str_count(raw$description, "\\S+")

In [8]:
cat("Records scraped:", nrow(raw), "\n")
print(table(raw$learning_modes))                           # shows the inconsistent labels
cat("Descriptions ending in '...':", sum(str_detect(raw$description, "\\.\\.\\.$")), "\n")
cat("Empty learning modes:", sum(raw$learning_modes == ""), "\n")
write.csv(raw, "Q4_1_raw.csv", row.names = FALSE)
print(head(raw[, c("title", "faculty", "programme_type", "n_modes", "desc_words", "start_date")]))

Records scraped: 65 

                                                         
                                                       1 
                                        Campus Full Time 
                                                       3 
    Campus Full Time, Online Full Time, Online Part Time 
                                                      40 
Distance (Online) Full Time, Distance (Online) Part-Time 
                                                       1 
                                      Distance Full Time 
                                                       1 
                  Distance Full Time, Distance Part Time 
                                                       1 
                      Online Full Time, Online Part Time 
                                                       8 
                                         Remote Learning 
                                                      10 
Descriptions ending in '...': 64 
Empty learning m

                                                      title             faculty
1                    Bachelor of Arts Access Programme: ENG Humanities and Arts
2                Bachelor of Arts Honours in Graphic Design Humanities and Arts
3                        Bachelor of Arts in Graphic Design Humanities and Arts
4 Bachelor of Arts: Industrial Psychology and HR Management Humanities and Arts
5                        Bachelor of Arts: Law and Politics Humanities and Arts
6                           Bachelor of Arts: Media Studies Humanities and Arts
  programme_type n_modes desc_words       start_date
1         Access       3         19 15 February 2027
2        Honours       2         24 15 February 2027
3         Degree       3         22 15 February 2027
4         Degree       3         25 15 February 2027
5         Degree       3         25 15 February 2027
6         Degree       3         25 15 February 2027


## Q4.2 Transform: clean, denoise and impute

In [9]:
clean <- raw
clean$start_text <- NULL

# denoise: remove the "..." the website adds when it cuts a description short
clean$description <- str_trim(str_remove(clean$description, "\\s*\\.\\.\\.$"))

# make the labels consistent: Distance and Distance (Online) are online, Part-Time is Part Time
clean$learning_modes <- str_replace_all(clean$learning_modes, "Distance \\(Online\\)|Distance", "Online")
clean$learning_modes <- str_replace_all(clean$learning_modes, "Part-Time", "Part Time")
clean$programme_type <- str_replace(clean$programme_type, "Postgraduate Certificates & Diplomas", "Postgraduate")

In [10]:
# missing value: an empty learning mode becomes NA and is imputed with the most
# common learning mode of the same programme type
clean$learning_modes[clean$learning_modes == ""] <- NA
most_common <- function(x) names(which.max(table(x)))
by_type <- tapply(clean$learning_modes, clean$programme_type, most_common)
missing <- is.na(clean$learning_modes)
clean$learning_modes[missing] <- by_type[clean$programme_type[missing]]

# recalculate the numerical columns on the cleaned text and fix the date type
clean$n_modes <- str_count(clean$learning_modes, "(?i)full time|part time|remote learning")
clean$desc_words <- str_count(clean$description, "\\S+")
clean$start_date <- as.Date(clean$start_date, format = "%d %B %Y")
clean <- unique(clean)                                      # remove duplicate records

In [11]:
cat("Records after cleaning:", nrow(clean), "\n")
print(table(clean$learning_modes))
cat("Missing values left:", sum(is.na(clean)), "\n")
write.csv(clean, "Q4_2_cleaned.csv", row.names = FALSE)
print(head(clean[, c("title", "faculty", "programme_type", "n_modes", "desc_words", "start_date")]))

Records after cleaning: 65 

                                    Campus Full Time 
                                                   3 
Campus Full Time, Online Full Time, Online Part Time 
                                                  41 
                                    Online Full Time 
                                                   1 
                  Online Full Time, Online Part Time 
                                                  10 
                                     Remote Learning 
                                                  10 
Missing values left: 0 
                                                      title             faculty
1                    Bachelor of Arts Access Programme: ENG Humanities and Arts
2                Bachelor of Arts Honours in Graphic Design Humanities and Arts
3                        Bachelor of Arts in Graphic Design Humanities and Arts
4 Bachelor of Arts: Industrial Psychology and HR Management Humanities and Arts
5      

## Q4.3 Load: PostgreSQL
The table, the import and the two analytical queries are in `Q4_3_postgres.sql`. The results are shown in the project document.